# Thread, Asyncio 를 사용한 동시 요청의 차이


In [ ]:
# server.py - 간단한 TCP 서버
import socket
import threading
import time


def handle_client(client_socket, address):  # 클라인트의 접속요청이 왔을 때 호출
    """클라이언트 요청을 처리하는 함수"""
    try:
        while True:
            data = client_socket.recv(1024).decode(
                "utf-8"
            )  # 클라이언트의 메시지가 수신되었을 때
            if not data:
                break

            print(f"[서버] {address}에서 받은 메시지: {data}")

            # 처리 시간 시뮬레이션 (1-3초)
            import random

            time.sleep(random.uniform(1, 3))  # 무작위 시간동안(1~3초) 쉰다

            # 클라이언트에게 응답 전송
            response = f"서버 응답: {data}를 처리했습니다"
            client_socket.send(response.encode("utf-8"))  # 클라이언트에게 메시지 전송

    except Exception as e:
        print(f"[서버] 오류 발생: {e}")
    finally:
        client_socket.close()
        print(f"[서버] {address} 연결 종료")


def start_server():
    """서버 시작"""
    # socket.AF_INET: 주소 체계로 IPv4를 사용, socket.SOCK_STREAM: 소켓 타입으로 TCP를 사용
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    # socket.SOL_SOCKET: 소켓 레벨의 옵션을 설정하겠다는 의미
    # socket.SO_REUSEADDR: 소켓이 이전에 사용했던 주소와 포트를 재사용할 수 있도록 설정
    # 이는 서버를 재시작할 때 "Address already in use" 오류를 방지하는 데 유용
    server.setsockopt(
        socket.SOL_SOCKET, socket.SO_REUSEADDR, 1
    )  # 1은 이 옵션을 활성화하겠다는 설정
    server.bind(("localhost", 8888))  # ('localhost', 8888): 바인딩할 주소와 포트
    server.listen(5)  # 5: 대기 큐의 크기를 지정. 그 이상은 연결 거부될 수 있음

    print("[서버] 포트 8888에서 서버 시작...")

    try:
        while (
            True
        ):  # 접속 요청이 오면 쓰레드를 생성하여 실행(접속 클라이언트 당 1개의 쓰레드가 생성됨)
            client_socket, address = server.accept()  # 클라이언트 접속 요청이 오면...
            print(f"[서버] {address}에서 연결됨")

            # 각 클라이언트를 별도 스레드에서 처리
            client_thread = threading.Thread(  # 쓰레드 생성
                target=handle_client,  # 쓰레드.start() 호출시 실행될 함수
                args=(client_socket, address),  # 쓰레드 함수로 전달될 아규먼트
            )
            client_thread.daemon = (
                True  # 메인 쓰레드가 죽으면 함께 죽는 데몬 쓰레드로 설정
            )
            client_thread.start()  # 쓰레드 실행 -> 쓰레드 함수 실행됨

    except KeyboardInterrupt:
        print("\n[서버] 서버 종료")
    finally:
        server.close()


if __name__ == "__main__":
    start_server()

[서버] 포트 8888에서 서버 시작...


In [ ]:
# client.py - 비동기 동시 요청 클라이언트
import socket
import threading
import time
import asyncio


class AsyncClient:
    def __init__(self, server_host="localhost", server_port=8888):
        self.server_host = server_host
        self.server_port = server_port

    def send_request(self, message, client_id):
        """단일 요청을 보내는 함수"""
        try:
            # 소켓 연결
            client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            client_socket.connect(
                (self.server_host, self.server_port)
            )  # 클라이언트가 서버측에 접속 요청을 보낸다

            # 요청 전송
            print(f"[클라이언트 {client_id}] 요청 전송: {message}")
            client_socket.send(
                message.encode("utf-8")
            )  # 클라이언트가 서버에 메시지 전송

            # 응답 수신
            response = client_socket.recv(1024).decode(
                "utf-8"
            )  # 클라이언트가 서버측에서 온 데이터(응답) 수신
            print(f"[클라이언트 {client_id}] 응답 받음: {response}")

            client_socket.close()  # 소켓 종료

        except Exception as e:
            print(f"[클라이언트 {client_id}] 오류 발생: {e}")

    def send_concurrent_requests_threading(
        self, messages
    ):  # 클라이언트가 서버에 대해서 쓰레드를 사용한 동시 요청
        """스레딩을 사용한 동시 요청"""
        print("\n=== 스레딩을 사용한 동시 요청 ===")
        start_time = time.time()

        threads = []
        for i, message in enumerate(
            messages
        ):  # 전달할 메시지의 수만큼 반복하여 쓰레드 생성 및 실행(메시지가 동시에 병행처리됨)
            thread = threading.Thread(target=self.send_request, args=(message, i + 1))
            threads.append(thread)
            thread.start()

        # 모든 스레드 완료 대기
        for thread in threads:
            thread.join()  # 데몬 쓰레드는 메인 쓰레드가 종료되면 따라 종료되므로 메인 쓰레드를 대기 상태로 설정

        end_time = time.time()  # 모든 쓰레드가 종료된 후에 실행됨
        print(f"총 소요 시간: {end_time - start_time:.2f}초\n")

    # coroutine 선언 : io 작업 등으로 인해 함수가 지연되는 경우에
    async def async_send_request(
        self, message, client_id
    ):  # await 키워드를 가진 호출을 포함한 경우에는 반드시 async 사용
        """asyncio를 사용한 비동기 요청"""
        try:
            # 비동기 소켓 연결
            reader, writer = (
                await asyncio.open_connection(  # 비동기 소켓을 이용한 접속 요청
                    self.server_host, self.server_port
                )
            )

            # 요청 전송
            print(f"[비동기 클라이언트 {client_id}] 요청 전송: {message}")
            writer.write(message.encode("utf-8"))
            await writer.drain()  # 전송 버퍼의 내용 전송 및 전송 완료시까지 기다림. 리턴될 때까지 기다리려면 반드시 await 키워드를 사용

            # 응답 수신
            response = await reader.read(1024)
            print(
                f"[비동기 클라이언트 {client_id}] 응답 받음: {response.decode('utf-8')}"
            )

            writer.close()
            await writer.wait_closed()

        except Exception as e:
            print(f"[비동기 클라이언트 {client_id}] 오류 발생: {e}")

    async def send_concurrent_requests_async(self, messages):
        """asyncio를 사용한 동시 요청"""
        print("=== asyncio를 사용한 동시 요청 ===")
        start_time = time.time()

        # 모든 요청을 동시에 실행
        tasks = [
            self.async_send_request(
                message, i + 1
            )  # coroutine객체만 생성되므로 로직 실행을 기다리지 않음
            for i, message in enumerate(messages)
        ]

        await asyncio.gather(
            *tasks
        )  # tasks에는 코루틴 객체가 포함되며 각각의 코루틴이 실행완료 될 때까지 기다림

        end_time = time.time()
        print(f"총 소요 시간: {end_time - start_time:.2f}초\n")


# 클라이언트 실행 예제
def main():
    client = AsyncClient()

    # 테스트 메시지들
    messages = [
        "요청 1: 데이터 조회",
        "요청 2: 파일 업로드",
        "요청 3: 계산 처리",
        "요청 4: 이메일 발송",
        "요청 5: 백업 실행",
    ]

    print("서버가 실행 중인지 확인하고 Enter를 누르세요...")
    input()

    # 1. 스레딩을 사용한 동시 요청
    client.send_concurrent_requests_threading(messages)

    time.sleep(2)  # 잠깐 대기

    # 2. asyncio를 사용한 동시 요청
    asyncio.run(client.send_concurrent_requests_async(messages))


#
if __name__ == "__main__":
    main()